# ETL: Extracción, transformación y carga

## 1. Librerías

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 2. Carga del dataset

In [2]:
df = pd.read_csv('../data/raw/ai_student_impact.csv')
print('Filas y columnas:', df.shape)
df.head()

Filas y columnas: (50000, 16)


,Student_ID,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.44,High
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.93,Medium
3,100004,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium
4,100005,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium


El dataset contiene 50,000 registros y 16 columnas. Cada fila representa un estudiante universitario con información sobre su perfil académico, sus hábitos de uso de herramientas de IA generativa durante un semestre y sus resultados al final del mismo. Es un dataset de tamaño considerable que permite obtener conclusiones estadísticamente robustas.

## 3. Inspección inicial

In [3]:
print('Tipos de datos')
print(df.dtypes)
print()
print('Valores nulos por columna')
print(df.isnull().sum())

=== Tipos de datos ===
Student_ID                      int64
Major_Category                    str
Year_of_Study                     str
Pre_Semester_GPA              float64
Weekly_GenAI_Hours            float64
Primary_Use_Case                  str
Prompt_Engineering_Skill          str
Tool_Diversity                  int64
Paid_Subscription                bool
Traditional_Study_Hours       float64
Perceived_AI_Dependency         int64
Institutional_Policy              str
Anxiety_Level_During_Exams      int64
Post_Semester_GPA             float64
Skill_Retention_Score         float64
Burnout_Risk_Level                str
dtype: object

=== Valores nulos por columna ===
Student_ID                    0
Major_Category                0
Year_of_Study                 0
Pre_Semester_GPA              0
Weekly_GenAI_Hours            0
Primary_Use_Case              0
Prompt_Engineering_Skill      0
Tool_Diversity                0
Paid_Subscription             0
Traditional_Study_Hours       0


In [4]:
print('Duplicados')
print('Filas duplicadas:', df.duplicated().sum())

Duplicados
Filas duplicadas: 0


In [5]:
print('Estadísticas descriptivas (variables numéricas)')
df.describe().round(2)

Estadísticas descriptivas (variables numéricas)


,Student_ID,Pre_Semester_GPA,Weekly_GenAI_Hours,Tool_Diversity,Traditional_Study_Hours,Perceived_AI_Dependency,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score
count,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00
mean,125000.50,3.15,8.43,2.80,11.21,3.51,4.27,3.35,75.80
std,14433.90,0.48,8.27,1.19,5.16,1.82,2.14,0.50,13.28
min,100001.00,1.18,0.00,1.00,1.00,1.00,1.00,1.00,10.78
25%,112500.75,2.83,2.39,2.00,7.56,2.00,3.00,3.02,66.82
50%,125000.50,3.21,5.80,3.00,11.18,3.00,4.00,3.42,76.00
75%,137500.25,3.52,11.72,4.00,14.71,5.00,6.00,3.75,85.19
max,150000.00,4.00,40.00,5.00,35.86,10.00,10.00,4.00,100.00


El dataset está en muy buenas condiciones: no hay valores nulos en ninguna columna ni registros duplicados. Los rangos de las variables numéricas son coherentes: el GPA oscila entre 1.18 y 4.0, las horas semanales de IA van de 0 a 40, y el puntaje de retención de habilidades cubre toda la escala de 10 a 100. Esto indica que el dataset no requiere limpieza agresiva, sino principalmente transformación y enriquecimiento.

In [6]:
print('Distribución de variables categóricas')
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df[col].value_counts())

Distribución de variables categóricas

--- Major_Category ---
Major_Category
STEM          15059
Business      12538
Humanities     9994
Medical        6476
Arts           5933
Name: count, dtype: int64

--- Year_of_Study ---
Year_of_Study
Junior       11045
Freshman     11031
Senior       10634
Sophomore     9860
Graduate      7430
Name: count, dtype: int64

--- Primary_Use_Case ---
Primary_Use_Case
Debugging/Troubleshooting    12295
Copywriting/Drafting         12011
Ideation                     10721
Summarizing_Reading           8633
Direct_Answer_Generation      6340
Name: count, dtype: int64

--- Prompt_Engineering_Skill ---
Prompt_Engineering_Skill
Beginner        18495
Intermediate    17696
Advanced        13809
Name: count, dtype: int64

--- Institutional_Policy ---
Institutional_Policy
Allowed_With_Citation    25224
Actively_Encouraged      14988
Strict_Ban                9788
Name: count, dtype: int64

--- Burnout_Risk_Level ---
Burnout_Risk_Level
Medium    21144
Low       1

Las variables categóricas muestran una distribución razonable entre sus categorías. La carrera más representada es STEM con aproximadamente 15,000 estudiantes (30%), seguida de Business (25%) y Humanities (20%). El año de estudio está bastante balanceado entre Freshman, Sophomore, Junior y Senior, con una porción adicional de Graduate students (15%). En cuanto a la política institucional, la mayoría de los estudiantes pertenece a instituciones que permiten el uso de IA con cita (Allowed_With_Citation), y aproximadamente el 30% proviene de instituciones que lo prohíben activamente (Strict_Ban). El nivel de burnout muestra que Medium es el más frecuente, seguido de Low y High.

## 4. Ajuste de tipos de datos

In [7]:
# Convertir Paid_Subscription de booleano a entero
df['Paid_Subscription'] = df['Paid_Subscription'].astype(int)

# Verificar rangos lógicos de las variables numéricas clave
print('Pre GPA  — min:', df['Pre_Semester_GPA'].min(), '| max:', df['Pre_Semester_GPA'].max())
print('Post GPA — min:', df['Post_Semester_GPA'].min(), '| max:', df['Post_Semester_GPA'].max())
print('Horas IA — min:', df['Weekly_GenAI_Hours'].min(), '| max:', df['Weekly_GenAI_Hours'].max())
print('Retención — min:', df['Skill_Retention_Score'].min(), '| max:', df['Skill_Retention_Score'].max())

Pre GPA  — min: 1.183 | max: 3.998
Post GPA — min: 1.0 | max: 4.0
Horas IA — min: 0.0 | max: 40.0
Retención — min: 10.78 | max: 100.0


Se convirtió la columna Paid_Subscription de booleano (True/False) a entero (1/0) para facilitar su uso en correlaciones y modelos numéricos. Todos los rangos verificados están dentro de los límites esperados.

## 5. Creación de variables derivadas

In [8]:
# Cambio en el GPA entre inicio y final del semestre
df['Cambio_GPA'] = (df['Post_Semester_GPA'] - df['Pre_Semester_GPA']).round(3)

print('Cambio_GPA — media:', df['Cambio_GPA'].mean().round(3))
print('Estudiantes que mejoraron:', (df['Cambio_GPA'] > 0).sum())
print('Estudiantes que empeoraron:', (df['Cambio_GPA'] < 0).sum())

Cambio_GPA — media: 0.203
Estudiantes que mejoraron: 43759
Estudiantes que empeoraron: 6192


El Cambio_GPA mide la variación del promedio académico a lo largo del semestre. Un valor positivo indica mejora y un valor negativo, deterioro. Esta variable es la medida más directa del impacto académico del semestre y será la variable objetivo del modelo de regresión en la fase 3.

In [9]:
# Nivel de uso de IA según horas semanales
def clasificar_uso_ia(horas):
    if horas < 5:
        return 'Bajo'
    elif horas < 15:
        return 'Medio'
    else:
        return 'Alto'

df['Nivel_Uso_IA'] = df['Weekly_GenAI_Hours'].apply(clasificar_uso_ia)
print(df['Nivel_Uso_IA'].value_counts())

Nivel_Uso_IA
Bajo     22567
Medio    18891
Alto      8542
Name: count, dtype: int64


El Nivel_Uso_IA clasifica a los estudiantes en tres grupos según sus horas semanales de uso: Bajo (menos de 5 horas), Medio (entre 5 y 15 horas) y Alto (más de 15 horas). Esta segmentación permitirá comparaciones grupales claras durante el análisis exploratorio.

In [10]:
# Dependencia alta: estudiantes con score de dependencia >= 7
df['Dependencia_Alta'] = (df['Perceived_AI_Dependency'] >= 7).astype(int)
print('Estudiantes con dependencia alta:', df['Dependencia_Alta'].sum())
print('Porcentaje:', round(df['Dependencia_Alta'].mean() * 100, 1), '%')

Estudiantes con dependencia alta: 3145
Porcentaje: 6.3 %


Se define Dependencia_Alta como un indicador binario que marca a los estudiantes con una dependencia percibida de 7 o más en la escala de 1 a 10. Este umbral corresponde al cuartil superior de la variable y permite identificar el subgrupo con mayor dependencia para análisis específicos.

In [11]:
# Horas totales de estudio (tradicional + IA)
df['Estudio_Total'] = (df['Traditional_Study_Hours'] + df['Weekly_GenAI_Hours']).round(2)
print('Estudio_Total — media:', df['Estudio_Total'].mean().round(2), 'horas/semana')

Estudio_Total — media: 19.64 horas/semana


El Estudio_Total suma las horas de estudio convencional con las horas de uso de IA por semana. Esta variable permite analizar si los estudiantes que usan más IA compensan reduciendo su estudio tradicional, o si ambas actividades coexisten sin reemplazarse mutuamente.

In [12]:
# Riesgo académico: GPA post bajo o caída significativa en el GPA
df['Riesgo_Academico'] = (
    (df['Post_Semester_GPA'] < 2.5) | (df['Cambio_GPA'] < -0.3)
).astype(int)

print('Estudiantes en riesgo académico:', df['Riesgo_Academico'].sum())
print('Porcentaje:', round(df['Riesgo_Academico'].mean() * 100, 1), '%')

Estudiantes en riesgo académico: 3411
Porcentaje: 6.8 %


El Riesgo_Academico es una variable binaria que marca como 'en riesgo' (1) a los estudiantes cuyo GPA post-semestre fue menor a 2.5, o cuyo GPA cayó más de 0.3 puntos durante el período. Esta será la variable objetivo del modelo de clasificación en la fase 3, y representa uno de los KPIs centrales del proyecto.

## 6. Encoding de variables categóricas

In [13]:
# Año de estudio a ordinal (de menor a mayor nivel académico)
year_map = {'Freshman': 1, 'Sophomore': 2, 'Junior': 3, 'Senior': 4, 'Graduate': 5}
df['Year_Encoded'] = df['Year_of_Study'].map(year_map)

# Habilidad en prompts a ordinal
skill_map = {'Beginner': 1, 'Intermediate': 2, 'Advanced': 3}
df['Prompt_Skill_Encoded'] = df['Prompt_Engineering_Skill'].map(skill_map)

# Política institucional a ordinal (de más restrictiva a más permisiva)
policy_map = {'Strict_Ban': 0, 'Allowed_With_Citation': 1, 'Actively_Encouraged': 2}
df['Policy_Encoded'] = df['Institutional_Policy'].map(policy_map)

# Burnout a ordinal
burnout_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['Burnout_Encoded'] = df['Burnout_Risk_Level'].map(burnout_map)

# Nivel de uso IA a ordinal
uso_map = {'Bajo': 0, 'Medio': 1, 'Alto': 2}
df['Uso_IA_Encoded'] = df['Nivel_Uso_IA'].map(uso_map)

print('Encodings aplicados correctamente.')
df[['Year_of_Study','Year_Encoded','Prompt_Engineering_Skill','Prompt_Skill_Encoded',
    'Institutional_Policy','Policy_Encoded','Burnout_Risk_Level','Burnout_Encoded']].head(5)

Encodings aplicados correctamente.


,Year_of_Study,Year_Encoded,Prompt_Engineering_Skill,Prompt_Skill_Encoded,Institutional_Policy,Policy_Encoded,Burnout_Risk_Level,Burnout_Encoded
0,Senior,4,Beginner,1,Allowed_With_Citation,1,High,2
1,Junior,3,Advanced,3,Allowed_With_Citation,1,Low,0
2,Freshman,1,Beginner,1,Strict_Ban,0,Medium,1
3,Senior,4,Intermediate,2,Allowed_With_Citation,1,Medium,1
4,Sophomore,2,Advanced,3,Allowed_With_Citation,1,Medium,1


Se aplicó encoding ordinal a todas las variables categóricas que tienen un orden natural inherente: el año de estudio (Freshman=1 hasta Graduate=5), la habilidad de prompting (Beginner=1 hasta Advanced=3), la política institucional (de prohibición total a uso activamente fomentado) y el nivel de burnout (de bajo a alto). Para Major_Category y Primary_Use_Case, que son nominales sin jerarquía, el one-hot encoding se aplicará directamente en el notebook de modelado según las necesidades de cada algoritmo.

## 7. Resumen del dataset transformado

In [14]:
print('Dimensiones finales del dataset procesado:', df.shape)
print()
df[['Student_ID','Pre_Semester_GPA','Post_Semester_GPA','Cambio_GPA',
    'Nivel_Uso_IA','Dependencia_Alta','Estudio_Total','Riesgo_Academico']].head(8)

Dimensiones finales del dataset procesado: (50000, 26)



,Student_ID,Pre_Semester_GPA,Post_Semester_GPA,Cambio_GPA,Nivel_Uso_IA,Dependencia_Alta,Estudio_Total,Riesgo_Academico
0,100001,2.418,2.393,-0.025,Alto,0,31.44,1
1,100002,3.821,3.696,-0.125,Bajo,0,17.77,0
2,100003,3.398,3.499,0.101,Alto,0,31.61,0
3,100004,3.789,4.000,0.211,Bajo,0,17.05,0
4,100005,3.635,3.798,0.163,Medio,0,21.84,0
5,100006,3.449,3.666,0.217,Medio,0,20.69,0
6,100007,3.622,4.000,0.378,Alto,1,44.52,0
7,100008,2.746,2.965,0.219,Medio,0,23.78,0


## 8. Exportación de archivos

In [15]:
# Dataset limpio: columnas originales transformadas, sin variables derivadas
cols_cleaned = [
    'Student_ID','Major_Category','Year_of_Study','Pre_Semester_GPA',
    'Weekly_GenAI_Hours','Primary_Use_Case','Prompt_Engineering_Skill',
    'Tool_Diversity','Paid_Subscription','Traditional_Study_Hours',
    'Perceived_AI_Dependency','Institutional_Policy','Anxiety_Level_During_Exams',
    'Post_Semester_GPA','Skill_Retention_Score','Burnout_Risk_Level'
]
df_cleaned = df[cols_cleaned].copy()
df_cleaned.to_csv('../data/cleaned/students_cleaned.csv', index=False)
print('students_cleaned.csv exportado:', df_cleaned.shape)

students_cleaned.csv exportado: (50000, 16)


In [16]:
# Dataset completo con variables derivadas y encodings, listo para modelado
df.to_csv('../data/processed/students_model_ready.csv', index=False)
print('students_model_ready.csv exportado:', df.shape)

students_model_ready.csv exportado: (50000, 26)


Se exportaron dos archivos. El archivo students_cleaned.csv conserva únicamente las columnas originales ya transformadas y es el punto de partida del análisis exploratorio. El archivo students_model_ready.csv incluye todas las variables derivadas y los encodings numéricos necesarios para alimentar los algoritmos de machine learning.

## Hallazgos de esta fase y conexión con el EDA

El dataset real está en condiciones excelentes: sin nulos, sin duplicados y con rangos coherentes en todas las variables. Las cinco variables derivadas construidas aquí serán el eje del análisis exploratorio:

El **Cambio_GPA** revelará si el semestre fue positivo o negativo para cada estudiante. El **Nivel_Uso_IA** permitirá comparar grupos de uso Bajo, Medio y Alto de manera directa. La **Dependencia_Alta** identificará el subgrupo de mayor riesgo potencial. El **Estudio_Total** permitirá evaluar si el uso de IA desplaza o complementa el estudio tradicional. Y el **Riesgo_Academico** será la variable objetivo de clasificación en la fase 3.

La decisión de usar encoding ordinal (en lugar de one-hot) para Year_of_Study, Prompt_Engineering_Skill, Institutional_Policy y Burnout_Risk_Level se tomó porque estas variables tienen un orden natural que conviene preservar en el análisis de correlaciones. El one-hot encoding para Major_Category y Primary_Use_Case queda reservado para el notebook de modelado donde cada algoritmo lo requerirá de forma específica.